# Inspect Consolidated Snow-Covered Area Dataset

Load and visualize the MOD10C1 v061 daily SCA fields for early March 2000.

Sources (see `catalog/variables.yml` → `snow_covered_area`):
- MOD10C1 v061 `Day_CMG_Snow_Cover` (percent, 0-100)
- MOD10C1 v061 `Snow_Spatial_QA` (percent, 0-100; CI threshold 70 applied downstream)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "2000-03-01"
TARGET_YEAR = 2000

## Load the dataset

In [ ]:
mod10c1_path = DATASTORE / "mod10c1_v061" / f"mod10c1_v061_{TARGET_YEAR}_consolidated.nc"

datasets = {
    "MOD10C1 v061 (Day_CMG_Snow_Cover)": {
        "path": mod10c1_path,
        "var": "Day_CMG_Snow_Cover",
        "units": "percent (0-100)",
        "cmap": "Blues",
    },
    "MOD10C1 v061 (Snow_Spatial_QA)": {
        "path": mod10c1_path,
        "var": "Snow_Spatial_QA",
        "units": "percent (0-100)",
        "cmap": "viridis",
    },
}

opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")

## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)

## Plot March 1, 2000

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = ds[var].sel(time=TARGET_TIME, method="nearest")
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap=info.get("cmap", "viridis"), robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"Snow-Covered Area — nearest to {TARGET_TIME}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()